In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from dotenv import load_dotenv
import os
from sklearn.utils.class_weight import compute_class_weight

# Load environment variables from the .env file
load_dotenv()

WORKSPACE_PATH = os.getenv("WORKSPACE_PATH")

# Add the parent directory to the system path
sys.path.append(str(WORKSPACE_PATH))

from src.config.config import DATA_DIR
from src.models.ml_metrics_utils import validate_model


Directory already exists: c:\Users\huber\OneDrive\Dokumenty\GitHub\ippan_suicide_study
Directory already exists: c:\Users\huber\OneDrive\Dokumenty\GitHub\ippan_suicide_study\data
Directory already exists: c:\Users\huber\OneDrive\Dokumenty\GitHub\ippan_suicide_study\results
Directory already exists: c:\Users\huber\OneDrive\Dokumenty\GitHub\ippan_suicide_study\results\plots
Directory already exists: c:\Users\huber\OneDrive\Dokumenty\GitHub\ippan_suicide_study\results\tables


In [2]:
from src.models.ml_metrics_utils import run_stratified_kfold

In [3]:
# Read CSV File
csv_file_path = DATA_DIR / "encoded" / "encoded_final_set.csv"
try:
    df_raw = pd.read_csv(csv_file_path, delimiter=",", low_memory=False)
except FileNotFoundError:
    print(f"Error: The file {csv_file_path} was not found.")
    sys.exit(1)

In [4]:
df_raw.head()

,Fatal,Gender,Context_Other,Context_FamilyConflict,Context_HeartBreak,Context_Finances,Context_SchoolWork,Context_CloseDeath,Context_Crime,Context_Disability,...,AgeGroup,AgeGroup2,CountContext,Date,DateM,DateY,Group_AF,Group_AG,Group_AGF,ID
0,False,True,True,True,False,False,False,False,False,False,...,25_29,19_34,2,2023-01-01,1.0,2023.0,19_34_0,19_34_M,19_34_M_0,138421942
1,False,True,False,False,False,False,False,False,False,False,...,55_59,35_64,1,2023-01-01,1.0,2023.0,35_64_0,35_64_M,35_64_M_0,138421944
2,False,False,False,True,False,False,False,False,False,False,...,13_18,00_18,1,2023-01-01,1.0,2023.0,00_18_0,00_18_F,00_18_F_0,138421949
3,False,True,False,True,False,False,False,False,False,False,...,55_59,35_64,1,2023-01-01,1.0,2023.0,35_64_0,35_64_M,35_64_M_0,138421953
4,False,False,False,False,False,False,True,False,False,False,...,19_24,19_34,1,2023-01-01,1.0,2023.0,19_34_0,19_34_F,19_34_F_0,138421954


In [5]:
# List of prefixes for selected columns
selected_prefixes = [
    "Income",
    "Method",
    "Education",
    "WorkInfo",
    "Substance",
    "Place",
    "Marital",
    "Context",
    "AbuseInfo",
    "Fatal",
]

group_column = "Group_AG"

selected_columns = [
    col
    for col in df_raw.columns
    if any(col.startswith(prefix) for prefix in selected_prefixes)
]
selected_columns.append(group_column)

df_data = df_raw[selected_columns].copy()

# Zamiana wartości "True"/"False" na 1/0 dla kolumn z wybranych prefiksów
for col in selected_columns:
    if col != group_column:  # Unikamy modyfikacji kolumny grupy
        df_data[col] = df_data[col].astype(int)

In [6]:
df_data.head()

,Fatal,Context_Other,Context_FamilyConflict,Context_HeartBreak,Context_Finances,Context_SchoolWork,Context_CloseDeath,Context_Crime,Context_Disability,Context_MentalHealth,...,Place_WaterRes,Place_Work,Marital_Cohabitant,Marital_Cohabiting,Marital_Divorced,Marital_Married,Marital_Separated,Marital_Single,Marital_Widowed,Group_AG
0,0,1,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,19_34_M
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,35_64_M
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,00_18_F
3,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,35_64_M
4,0,0,0,0,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,19_34_F


In [7]:
groups = sorted(list(set(df_data[group_column])))

In [8]:
group = groups[0]
group

'00_18_F'

In [9]:
df_model = df_data[df_data[group_column] == group]
df_model = df_model.drop(columns=[group_column])

In [10]:
df_model.shape

(5775, 63)

In [11]:
df_model.head()

,Fatal,Context_Other,Context_FamilyConflict,Context_HeartBreak,Context_Finances,Context_SchoolWork,Context_CloseDeath,Context_Crime,Context_Disability,Context_MentalHealth,...,Place_UtilitySpaces,Place_WaterRes,Place_Work,Marital_Cohabitant,Marital_Cohabiting,Marital_Divorced,Marital_Married,Marital_Separated,Marital_Single,Marital_Widowed
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
25,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
31,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
36,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
39,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0


In [12]:
# Prepare the target variable 'Y' and features 'X'
Y = df_model["Fatal"]  # Target variable (fatal or not)
X = df_model.drop(
    "Fatal", axis=1, errors="ignore"
)  # Features excluding the target variable

# Prepare the list of feature names
feature_names = X.columns.tolist()

In [13]:
# Compute class weights based on the entire dataset to handle inbalance in classes
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(Y), y=Y)
class_weights_dict = {cls: weight for cls, weight in zip(np.unique(Y), class_weights)}

In [14]:
# Initialize model
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    max_depth=None, min_samples_split=10, min_samples_leaf=10
)

In [15]:
param_grid = {
    "n_estimators": 100,
    "max_features": "sqrt",
    "max_depth": None,
    "min_samples_split": 10,
    "min_samples_leaf": 10,
}

In [16]:
# Perform stratified K-Fold validation
skf_results = run_stratified_kfold(model, X, Y, n_splits=5)

In [17]:
skf_results

,recall,accuracy,precision,f1_score,fold,train_size,test_size
0,0.935931,0.935931,0.918764,0.917811,1,4620,1155
1,0.931602,0.931602,0.908711,0.912793,2,4620,1155
2,0.935931,0.935931,0.920814,0.915045,3,4620,1155
3,0.936797,0.936797,0.928553,0.913278,4,4620,1155
4,0.933333,0.933333,0.921387,0.904335,5,4620,1155


In [18]:
# Calculate the mean of selected columns
skf_final = skf_results[["recall", "accuracy", "precision", "f1_score"]].mean()

# Convert the Series to a DataFrame and transpose it to make it a single row
skf_final = skf_final.to_frame().T

# Set a custom index label
skf_final.index = ["skf_result"]


In [19]:
# Fit the model
model.fit(X, Y)

# Validate the model
final_results = validate_model(model, X, Y)

# Add a custom index label
final_results.index = ["final_result"]

In [20]:
# Ensure both are DataFrames
if isinstance(skf_final, pd.Series):
    skf_final = skf_final.to_frame().T  # Convert Series to DataFrame with one row

if isinstance(final_results, pd.Series):
    final_results = (
        final_results.to_frame().T
    )  # Convert Series to DataFrame with one row

In [21]:
# Combine the DataFrames row-wise
results = pd.concat([skf_final, final_results], axis=0, ignore_index=False)


In [22]:
results

,recall,accuracy,precision,f1_score
skf_result,0.934719,0.934719,0.919646,0.912653
final_result,0.936450,0.936450,0.923644,0.914046


# Feature validation

In [24]:
from src.models.ml_metrics_utils import compute_permutation_importance

# Compute permutation importance
perm_importance = compute_permutation_importance(model, X, Y)


In [25]:
perm_importance

,feature,importance_mean,importance_std
0,Method_Hanging,0.003931,0.000300
1,Substance_OtherSub,0.002216,0.001189
2,Method_Drugs,0.002043,0.000589
3,Method_SelfHarm,0.001437,0.000514
4,Place_House,0.001264,0.000629
...,...,...,...
57,Context_CloseDeath,-0.000121,0.000079
58,Education_LowerSecondary,-0.000208,0.000104
59,Context_SchoolWork,-0.000208,0.000242
60,Context_MentalHealth,-0.000485,0.000327


In [26]:
from src.models.ml_metrics_utils import compute_mean_decrease_accuracy

# Compute mean decrease accuracy
mean_decrease_accuracy = compute_mean_decrease_accuracy(model, X, Y)


In [27]:
mean_decrease_accuracy

,feature,MDA
0,Context_Other,0.002251
1,Context_FamilyConflict,0.001212
2,Context_HeartBreak,0.000173
3,Context_Finances,0.000866
4,Context_SchoolWork,0.001385
...,...,...
57,Marital_Divorced,0.000346
58,Marital_Married,0.000519
59,Marital_Separated,0.000346
60,Marital_Single,-0.000346
